# Гипотеза 2. Вахтовый метод связан с более высокими зарплатными ожиданиями

Первая гипотеза про образование не дала устойчивого подтверждения. Следующий шаг — проверить фактор, который уже намекал в EDA на заметный эффект: предпочтение **вахтового метода**.

Проверяем гипотезу:

**H2:** соискатели, указавшие `schedule_type_1 = 1` (вахтовый метод), ожидают более высокую зарплату, чем остальные соискатели.

Критерий подтверждения:
- медианная зарплата в группе вахтового метода выше;
- 95% bootstrap CI для разницы медиан полностью выше 0;
- перестановочный тест подтверждает статистическую значимость;
- эффект сохраняется хотя бы в нескольких крупных региональных и опытных срезах.

In [1]:
import os
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path(".mplconfig"))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

rng = np.random.default_rng(42)

In [2]:
def load_clean_df():
    df = pd.read_csv("dataset_cleaned.csv", parse_dates=["date_publish"], low_memory=False)
    numeric_cols = [
        "salary",
        "experience",
        "birthday",
        "relocation",
        "business_trips",
        "retraining_capability",
        "inner_info_fullness_rate",
        "schedule_type_1",
        "schedule_type_2",
        "schedule_type_3",
        "schedule_type_4",
        "schedule_type_5",
        "schedule_type_6",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "region_group" in df.columns:
        df["region_primary_group"] = (
            df["region_group"]
            .astype("string")
            .str.split("|")
            .str[0]
        )
    if "date_publish" in df.columns and "birthday" in df.columns:
        birthday = pd.to_numeric(df["birthday"], errors="coerce")
        df["age"] = df["date_publish"].dt.year - birthday
        df.loc[~df["age"].between(14, 80), "age"] = np.nan
    return df


def bootstrap_ci_diff(x, y, stat_func=np.median, n_boot=2000, seed=42):
    rng_local = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        xb = rng_local.choice(x, size=len(x), replace=True)
        yb = rng_local.choice(y, size=len(y), replace=True)
        boot[i] = stat_func(xb) - stat_func(yb)
    return np.quantile(boot, [0.025, 0.975])


def permutation_pvalue(x, y, stat_func=np.median, n_perm=2000, seed=42):
    rng_local = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    observed = stat_func(x) - stat_func(y)
    pooled = np.concatenate([x, y])
    stats = np.empty(n_perm)
    x_len = len(x)
    for i in range(n_perm):
        shuffled = rng_local.permutation(pooled)
        stats[i] = stat_func(shuffled[:x_len]) - stat_func(shuffled[x_len:])
    pvalue = (np.abs(stats) >= abs(observed)).mean()
    return observed, pvalue

In [3]:
df = load_clean_df()

watch_df = df[df["schedule_type_1"].isin([0, 1])].copy()
watch_df["watch_shift"] = watch_df["schedule_type_1"].eq(1)
watch_df["exp_bucket"] = pd.cut(
    watch_df["experience"],
    bins=[-0.1, 1, 5, 10, 60],
    labels=["0-1", "2-5", "6-10", "11+"],
)

print("clean_df shape:", df.shape)
print("subset with known schedule_type_1:", watch_df.shape)
watch_df["watch_shift"].value_counts().rename(index={False: "other", True: "watch_shift"})

clean_df shape: (43227, 66)
subset with known schedule_type_1: (43227, 68)


watch_shift
other          42839
watch_shift      388
Name: count, dtype: int64

In [4]:
overall_summary = (
    watch_df.groupby("watch_shift")["salary"]
    .agg(count="size", median="median", mean="mean")
    .rename(index={False: "other", True: "watch_shift"})
)

region_summary = (
    watch_df.groupby(["region_primary_group", "watch_shift"])["salary"]
    .agg(count="size", median="median", mean="mean")
    .reset_index()
)

experience_summary = (
    watch_df.groupby(["exp_bucket", "watch_shift"], observed=False)["salary"]
    .agg(count="size", median="median", mean="mean")
    .reset_index()
)

display(overall_summary)
display(region_summary)
display(experience_summary)

,count,median,mean
watch_shift,,,
other,42839,20000.0,25084.161909
watch_shift,388,45000.0,49665.587629


,region_primary_group,watch_shift,count,median,mean
0,Дальний Восток,False,2604,20000.0,24779.317588
1,Дальний Восток,True,50,30000.0,40420.000000
2,Промышленные центры,False,20247,20000.0,23002.001136
3,Промышленные центры,True,180,50000.0,55666.666667
4,Столицы,False,4231,35000.0,38622.094777
5,Столицы,True,24,30099.0,38581.166667
6,Центральная Россия,False,9118,20000.0,25255.889888
7,Центральная Россия,True,33,30000.0,42424.242424
8,Южные регионы,False,6609,20000.0,22681.187018
9,Южные регионы,True,100,47500.0,48333.000000


,exp_bucket,watch_shift,count,median,mean
0,0-1,False,18301,20000.0,22792.738812
1,0-1,True,174,35000.0,43945.390805
2,2-5,False,9044,20000.0,24305.936975
3,2-5,True,77,40000.0,44126.623377
4,6-10,False,7248,25000.0,26328.660734
5,6-10,True,72,50000.0,57819.444444
6,11+,False,8240,25000.0,29916.076699
7,11+,True,65,50000.0,62507.692308


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=watch_df,
    x="watch_shift",
    y="salary",
    showfliers=False,
    ax=axes[0],
)
axes[0].set_title("Зарплатные ожидания: вахтовый метод vs остальные")
axes[0].set_xlabel("Вахтовый метод")
axes[0].set_ylabel("Желаемая зарплата")
axes[0].set_xticklabels(["Нет", "Да"])

region_plot = (
    region_summary[region_summary["watch_shift"]]
    .sort_values("median", ascending=False)
)
sns.barplot(
    data=region_plot,
    x="median",
    y="region_primary_group",
    ax=axes[1],
)
axes[1].set_title("Медианная зарплата в группе вахтового метода по регионам")
axes[1].set_xlabel("Медианная желаемая зарплата")
axes[1].set_ylabel("Региональная группа")

plt.tight_layout()
plt.show()

/var/folders/5y/j_6skz8d0y9ggl6xsh5j5zz80000gn/T/ipykernel_27852/1065766078.py:13: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0].set_xticklabels(["Нет", "Да"])
/var/folders/5y/j_6skz8d0y9ggl6xsh5j5zz80000gn/T/ipykernel_27852/1065766078.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
x = watch_df.loc[watch_df["watch_shift"], "salary"].dropna().to_numpy()
y = watch_df.loc[~watch_df["watch_shift"], "salary"].dropna().to_numpy()

median_diff_ci = bootstrap_ci_diff(x, y, stat_func=np.median, n_boot=2000, seed=42)
mean_diff_ci = bootstrap_ci_diff(x, y, stat_func=np.mean, n_boot=2000, seed=42)
observed_diff, pvalue = permutation_pvalue(x, y, stat_func=np.median, n_perm=2000, seed=42)

result = pd.Series({
    "n_watch_shift": len(x),
    "n_other": len(y),
    "median_watch_shift": float(np.median(x)),
    "median_other": float(np.median(y)),
    "median_diff": float(observed_diff),
    "median_diff_ci_low": float(median_diff_ci[0]),
    "median_diff_ci_high": float(median_diff_ci[1]),
    "mean_watch_shift": float(np.mean(x)),
    "mean_other": float(np.mean(y)),
    "mean_diff_ci_low": float(mean_diff_ci[0]),
    "mean_diff_ci_high": float(mean_diff_ci[1]),
    "permutation_pvalue": float(pvalue),
})

result

n_watch_shift            388.000000
n_other                42839.000000
median_watch_shift     45000.000000
median_other           20000.000000
median_diff            25000.000000
median_diff_ci_low     20000.000000
median_diff_ci_high    30000.000000
mean_watch_shift       49665.587629
mean_other             25084.161909
mean_diff_ci_low       21686.923713
mean_diff_ci_high      27646.907205
permutation_pvalue         0.000000
dtype: float64

In [7]:
robustness_region = (
    region_summary
    .pivot(index="region_primary_group", columns="watch_shift", values=["count", "median"])
    .sort_index()
)
robustness_region.columns = [
    "count_other",
    "count_watch",
    "median_other",
    "median_watch",
] if len(robustness_region.columns) == 4 else robustness_region.columns
robustness_region["median_diff"] = robustness_region["median_watch"] - robustness_region["median_other"]

robustness_experience = (
    experience_summary
    .pivot(index="exp_bucket", columns="watch_shift", values=["count", "median"])
    .sort_index()
)
robustness_experience.columns = [
    "count_other",
    "count_watch",
    "median_other",
    "median_watch",
] if len(robustness_experience.columns) == 4 else robustness_experience.columns
robustness_experience["median_diff"] = robustness_experience["median_watch"] - robustness_experience["median_other"]

display(robustness_region)
display(robustness_experience)

,count_other,count_watch,median_other,median_watch,median_diff
region_primary_group,,,,,
Дальний Восток,2604.0,50.0,20000.0,30000.0,10000.0
Промышленные центры,20247.0,180.0,20000.0,50000.0,30000.0
Столицы,4231.0,24.0,35000.0,30099.0,-4901.0
Центральная Россия,9118.0,33.0,20000.0,30000.0,10000.0
Южные регионы,6609.0,100.0,20000.0,47500.0,27500.0


,count_other,count_watch,median_other,median_watch,median_diff
exp_bucket,,,,,
0-1,18301.0,174.0,20000.0,35000.0,15000.0
2-5,9044.0,77.0,20000.0,40000.0,20000.0
6-10,7248.0,72.0,25000.0,50000.0,25000.0
11+,8240.0,65.0,25000.0,50000.0,25000.0


In [8]:
is_confirmed = (
    (result["median_diff"] > 0)
    and (result["median_diff_ci_low"] > 0)
    and (result["permutation_pvalue"] < 0.05)
)

if is_confirmed:
    print("Гипотеза подтверждена.")
    print("Эффект сильный: у группы с вахтовым методом медианная зарплата заметно выше.")
else:
    print("Гипотеза не подтверждена.")

Гипотеза подтверждена.
Эффект сильный: у группы с вахтовым методом медианная зарплата заметно выше.


## Вывод

Гипотеза подтверждается. В очищенной выборке соискатели, готовые к вахтовому методу, ожидают существенно более высокую зарплату, чем остальные.

Эффект виден не только в среднем по выборке, но и внутри нескольких крупных срезов по опыту и региональным группам. Это делает гипотезу содержательно полезной: формат занятости выступает важным маркером сегмента рынка труда с более высокими зарплатными ожиданиями.